# Phase0: 環境確認とデータ準備

**この回で学ぶこと**
- 教師データ（画像）の取得と、`train / val / test` への分割
- **データ分割は拡張の「前」に行う**理由（データリーク）

**使うデータ**: hymenoptera（アリ vs ハチ）。PyTorch公式チュートリアルの小さな2クラス画像データです。
少量データなので、データ拡張の効果が見えやすいのが利点です。

> ⏱ 所要時間: 1〜2分（初回のみダウンロードあり）

In [ ]:
import sys
from pathlib import Path

# src/ をimportできるようにする（notebooks/ からでもプロジェクト直下からでも動く）
sys.path.insert(0, str(next(p for p in [Path.cwd() / "src", Path.cwd().parent / "src"] if p.exists())))
from nb_utils import *

%matplotlib inline

In [ ]:
import torchvision
print("torch", torch.__version__, "/ torchvision", torchvision.__version__)
print("使用デバイス:", "mps (Apple Silicon GPU)" if torch.backends.mps.is_available() else "cpu")

## 1. データの取得と分割

`src/prepare_data.py` が次の処理をします（シード42固定で再現可能）。

| 分割 | 元 | 用途 |
|---|---|---|
| train | 公式の train | 学習に使う。**拡張はここだけ**に適用 |
| val | 公式の val の半分 | 学習中の様子を見る・条件比較 |
| test | 公式の val の残り半分 | 最後に1回だけ評価 |

In [ ]:
import prepare_data

src_dir = prepare_data.download()   # 取得済みならスキップ
prepare_data.split(src_dir)
prepare_data.summary()

## 2. データを眺める

モデルに学習させる前に、必ず自分の目でデータを確認します。ラベルの間違いや、写り方の偏りに気付けます。

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

classes = sorted(p.name for p in (DATA / "train").iterdir() if p.is_dir())
imgs, titles = [], []
for c in classes:
    for f in sorted((DATA / "train" / c).glob("*.jpg"))[:8]:
        imgs.append(Image.open(f).convert("RGB").resize((160, 160)))
        titles.append(c)
show_images(imgs, titles, cols=8, size=1.8);

In [ ]:
# 枚数の確認（クラス不均衡がないか）
counts = {part: {c: len(list((DATA / part / c).glob("*.jpg"))) for c in classes} for part in ("train", "val", "test")}
pd.DataFrame(counts)

## 3. データリークがないことの確認

**データリーク**とは、train にあるデータ（またはその派生）が val/test にも入ってしまい、評価が甘くなる問題です。
拡張で増やした画像を後から分割すると、「同じ元画像の反転版」が train と test に分かれてしまい、
**見たことのある画像でテスト**することになります。

だから「**分割 → 拡張**」の順にします。ここでは、ファイル名が分割をまたいで重複していないか確認します。

In [ ]:
names = {part: {f.name for f in (DATA / part).rglob("*.jpg")} for part in ("train", "val", "test")}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    dup = names[a] & names[b]
    print(f"{a} と {b} の重複: {len(dup)}件")
    assert not dup, "データリークの疑いあり"
print("重複なし ✔")

## 🧪 試してみよう
1. 上の画像表示で、`[:8]` を変えて別の画像も見てみよう。ラベルと見た目が合わないものはあるか？
2. `prepare_data.py` の `SEED` を変えて分割し直すと、val/test の枚数や中身はどう変わる？（実行後は元に戻してから次のNotebookへ）
3. 画像サイズ・向き・背景はそろっているか？ そろっていない要素は、拡張で補える可能性がある。

## まとめ
- 画像データは `train / val / test` に分け、**分割は拡張の前**に行う。
- 少量データ（train 243枚）なので、過学習しやすく、拡張の効果が見えやすい。

➡ 次は `01_visualize_augment.ipynb`（拡張で画像がどう変わるかを見る）